# Embedding pipeline
Create embeddings from product CSV, save locally, and (optionally) upload to Pinecone.


In [1]:
# Cell 1 — Install (run once if needed)
# Uncomment the lines you need. Running installs in notebooks can be slow.
# !pip install -U sentence-transformers==2.2.2 pinecone-client pandas tqdm faiss-cpu
# For OpenAI embeddings instead:
# !pip install openai


In [2]:
# Cell 2 — Config
import os
from pathlib import Path
CSV_PATH = os.getenv("CLEANED_CSV_PATH", "../data/clean_sample_with_imputed_prices.csv")  # The new CSV path
# Change the default value for the output directory
OUTPUT_DIR = Path(os.getenv("EMBED_OUT", "../data/embeddings")) # The new output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model options: "sentence-transformers/all-mpnet-base-v2" or "openai" (requires OPENAI_API_KEY)
TEXT_MODEL = os.getenv("TEXT_MODEL_NAME", "sentence-transformers/all-mpnet-base-v2")
USE_OPENAI = TEXT_MODEL.lower().startswith("openai")  # if you set TEXT_MODEL to "openai" you enable OpenAI path

# chunking / batching
CHUNK_THRESHOLD = int(os.getenv("CHUNK_THRESHOLD", "1200"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "200"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64"))

# Pinecone config (optional — used only if you run upload step)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_ENV = os.getenv("PINECONE_ENV")
PINECONE_INDEX = os.getenv("PINECONE_INDEX")
PINECONE_BATCH_UPSERT = int(os.getenv("PINECONE_BATCH_UPSERT", "100"))


In [3]:
# Cell 3 — Utilities: chunker and text composer
import re
import pandas as pd

def chunk_text(text: str, threshold=CHUNK_THRESHOLD, overlap=CHUNK_OVERLAP):
    if not isinstance(text, str):
        text = str(text or "")
    text = text.strip()
    if len(text) <= threshold:
        return [text]
    chunks = []
    start = 0
    L = len(text)
    while start < L:
        end = min(L, start + threshold)
        chunks.append(text[start:end].strip())
        if end == L:
            break
        start = max(0, end - overlap)
    return chunks

def make_combined_text(row,
                       cols=("title","brand","description","categories","material","color")):
    parts = []
    for c in cols:
        if c in row and pd.notna(row[c]):
            s = str(row[c]).strip()
            if s:
                parts.append(s)
    combined = " ".join(parts)
    # collapse whitespace
    combined = re.sub(r"\s+", " ", combined).strip()
    return combined


In [4]:
# Cell 4 — Read CSV and prepare entries
df = pd.read_csv(CSV_PATH)
# Ensure common text cols exist
for col in ["title","brand","description","categories","material","color","uniq_id","price_clean"]:
    if col not in df.columns:
        df[col] = ""

# generate combined_text if missing
if "combined_text" not in df.columns or df["combined_text"].isnull().any():
    df["combined_text"] = df.apply(make_combined_text, axis=1)

entries = []  # each entry: {"id": eid, "text": chunk_text, "meta": {...}}
for _, row in df.iterrows():
    pid = str(row.get("uniq_id", "") or "")
    text = row.get("combined_text", "")
    chunks = chunk_text(text)
    for i, ch in enumerate(chunks):
        eid = pid if len(chunks) == 1 else f"{pid}::chunk{i}"
        meta = {
            "uniq_id": pid,
            "title": row.get("title",""),
            "brand": row.get("brand",""),
            "price": row.get("price_clean", None),
            "category_list": row.get("category_list", []) if "category_list" in row else row.get("categories", ""),
            "chunk_index": i,
            "chunk_count": len(chunks),
            "combined_len": len(ch)
        }
        entries.append({"id": eid, "text": ch, "meta": meta})

len(entries)


341

In [5]:
# Cell 5 — Encode embeddings (Sentence-Transformers or OpenAI)
import numpy as np
from tqdm import tqdm

texts = [e["text"] for e in entries]
ids = [e["id"] for e in entries]
metas = [e["meta"] for e in entries]

if USE_OPENAI:
    # OpenAI embeddings path
    import openai
    openai.api_key = os.getenv("OPENAI_API_KEY")
    # choose model, e.g. text-embedding-3-large or text-embedding-3-small
    OPENAI_EMB_MODEL = os.getenv("OPENAI_EMB_MODEL", "text-embedding-3-small")
    def chunked(it, size):
        for i in range(0, len(it), size):
            yield it[i:i+size]
    all_embs = []
    for b in tqdm(list(chunked(texts, BATCH_SIZE)), desc="OpenAI batches"):
        resp = openai.embeddings.create(input=b, model=OPENAI_EMB_MODEL)
        emb_batch = [r["embedding"] for r in resp["data"]]
        all_embs.extend(emb_batch)
    embeddings = np.array(all_embs, dtype=np.float32)
else:
    # Sentence-Transformers path (local)
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(TEXT_MODEL)
    all_embs = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="ST batches"):
        batch = texts[i:i+BATCH_SIZE]
        emb = model.encode(batch, show_progress_bar=True, convert_to_numpy=True)
        all_embs.append(emb)
    embeddings = np.vstack(all_embs)

# normalize for cosine similarity if you plan to use cosine
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms==0] = 1.0
embeddings = embeddings / norms
embeddings = embeddings.astype(np.float32)
embeddings.shape


ST batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

ST batches:  17%|█▋        | 1/6 [00:01<00:05,  1.08s/it]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

ST batches:  33%|███▎      | 2/6 [00:01<00:03,  1.10it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

ST batches:  83%|████████▎ | 5/6 [00:04<00:00,  1.23it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

ST batches: 100%|██████████| 6/6 [00:04<00:00,  1.28it/s]


(341, 768)

In [6]:
# Cell 6 — Save embeddings + metadata locally
import json
np.savez_compressed(OUTPUT_DIR / "embeddings.npz", embeddings=embeddings, ids=np.array(ids))
# metadata as jsonl (one per line)
with open(OUTPUT_DIR / "metadata.jsonl", "w", encoding="utf-8") as f:
    for i, m in zip(ids, metas):
        out = {"id": i, "metadata": m}
        f.write(json.dumps(out) + "\n")

print("Saved:", OUTPUT_DIR)


Saved: ..\data\embeddings


In [7]:
# Cell 7 — Optional: quick FAISS save (fallback) — produces index file + meta
try:
    import faiss
    d = embeddings.shape[1]
    # use inner product on normalized vectors -> cosine
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)
    faiss.write_index(index, str(OUTPUT_DIR / "faiss_index.bin"))
    import pickle
    with open(OUTPUT_DIR / "faiss_index.meta.pkl", "wb") as f:
        pickle.dump({"ids": ids, "metadata": metas}, f)
    print("FAISS index saved.")
except Exception as e:
    print("FAISS not available or failed:", e)


FAISS index saved.


In [8]:
# # Cell 8 — Optional: Upload to Pinecone (run only if you want to push now)
# # Make sure PINECONE_API_KEY and PINECONE_ENV are set in env vars
# if PINECONE_API_KEY:
#     import pinecone
#     pinecone.init(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)
#     if PINECONE_INDEX not in pinecone.list_indexes():
#         pinecone.create_index(PINECONE_INDEX, dimension=embeddings.shape[1], metric="cosine")
#     idx = pinecone.Index(PINECONE_INDEX)
#     # upsert in batches
#     batch_size = PINECONE_BATCH_UPSERT
#     to_upsert = []
#     for i in range(len(ids)):
#         to_upsert.append((ids[i], embeddings[i].tolist(), metas[i]))
#         if len(to_upsert) >= batch_size:
#             idx.upsert(vectors=to_upsert)
#             to_upsert = []
#     if to_upsert:
#         idx.upsert(vectors=to_upsert)
#     print("Uploaded to Pinecone index:", PINECONE_INDEX)
# else:
#     print("Pinecone API key not set; skipping upload. Saved artifacts ready for upload.")


In [9]:
# Cell 8 — Optional: Upload to Pinecone (run only if you want to push now)
# Make sure PINECONE_API_KEY and PINECONE_ENV are set in env vars

import math


if PINECONE_API_KEY:
    # 1. Update the import to get the Pinecone class
    from pinecone import Pinecone, ServerlessSpec
    
    # 2. Initialize the client (replaces pinecone.init())
    # NOTE: The 'environment' parameter is deprecated and no longer needed here.
    # The region is now specified within the index creation 'spec'.
    pc = Pinecone(api_key=PINECONE_API_KEY)

    PINECONE_REGION = os.getenv("PINECONE_ENV") # You must define this

    # 3. Check for index existence and create it using the new syntax
    if PINECONE_INDEX not in pc.list_indexes().names():
        print(f"Creating index '{PINECONE_INDEX}'...")
        
        # Use ServerlessSpec (or PodSpec if you prefer pods)
        # Assuming Serverless is the modern choice. Replace 'aws' and the region 
        # with your desired cloud and region.
        pc.create_index(
            name=PINECONE_INDEX, 
            dimension=embeddings.shape[1], 
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region=PINECONE_REGION)
        )
        print("Index created successfully.")
        
    # 4. Connect to the index using the new client instance (pc)
    idx = pc.Index(PINECONE_INDEX)
    
    # upsert in batches
    batch_size = PINECONE_BATCH_UPSERT
    to_upsert = []
    for i in range(len(ids)):
        current_meta = metas[i]
        cleaned_meta={}
        for k, v in current_meta.items():
            is_nan=False
            try:
                if isinstance(v,float) and math.isnan(v):
                    is_nan=True
            except TypeError:
                pass
            if not is_nan:
                cleaned_meta[k]=v
            else:
                cleaned_meta[k]=None
                
        to_upsert.append((ids[i], embeddings[i].tolist(), cleaned_meta))
        if len(to_upsert) >= batch_size:
            # Upsert using the index object 'idx'
            idx.upsert(vectors=to_upsert)
            to_upsert = []
    
    if to_upsert:
        idx.upsert(vectors=to_upsert)
        
    print("Uploaded to Pinecone index:", PINECONE_INDEX)
else:
    print("Pinecone API key not set; skipping upload. Saved artifacts ready for upload.")

Uploaded to Pinecone index: ikarus-products


In [10]:
# Cell 9 — Small verification: nearest neighbor sanity-check (local)
# Use FAISS or brute-force for a tiny sample
query_idx = 0
q = embeddings[query_idx]
# brute-force cosine similarity
scores = embeddings @ q
topk = 5
topk_idx = np.argsort(-scores)[:topk]
for r in topk_idx:
    print(r, ids[r], metas[r]["title"][:120], float(scores[r]))


0 02593e81-5c09-5069-8516-b0b29f439ded goymfk 1pc free standing shoe rack, multi-layer metal shoe cap rack with 8 double hooks for living room, bathroom, hallw 1.0
7 02593e81-5c09-5069-8516-b0b29f439ded goymfk 1pc free standing shoe rack, multi-layer metal shoe cap rack with 8 double hooks for living room, bathroom, hallw 1.0
90 122c5c2a-5490-51ce-8555-9526c9698a38 lanteful shoe rack organizer shoe storage cabinet 8 tiers 32 pair portable shoe storage sturdy plastic black shoe shelf  0.750805675983429
168 f28d5cba-ecd4-5d82-87da-d926d48e1155 sogesfurniture 5 tier free standing wooden shoe storage shelf shoe organizer, 29.5 inches shoe rack shoe organizer stora 0.7119339108467102
199 38587d05-ed7c-54eb-acc2-057cec374f51 dscabomlg foldable shoe storage plastic vertical shoe rack shoe organizer for closet narrow shoe shelf plastic-b 0.6943106651306152
